# Pyrates FEP
# Notebook 05: Full campaign, results analysis

Inter-rater reliability of the full campaign coding (8 experts). Each feedback is coded by two
experts, but the pair changes from game to game (16 different pairs): agreement is therefore
measured with Krippendorff's alpha, which does not assume a fixed pair of coders, with one main
measure per item: ordinal alpha for the resolution stage, alpha on valid / invalid for the
invalidity, MASI alpha on the whole answer for the content and the instructions, completed by a
per-code analysis. Confidence intervals are bootstrapped by game, since the feedback of one game are
not independent. The last section gathers the main measure of each item.

### Inputs:
- `campaign/2_full/full_form.json`
- `campaign/2_full/full_campaign_assignment.csv` (notebook 04)
- `campaign/2_full/full_result.csv` (full campaign coding, one row per (game, trace, expert))

## 1. Setup and imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CAMPAIGN_DIR = PROJECT_ROOT / "campaign" / "2_full"
FORM_PATH = CAMPAIGN_DIR / "full_form.json"
ASSIGNMENT_PATH = CAMPAIGN_DIR / "full_campaign_assignment.csv"
RESULT_PATH = CAMPAIGN_DIR / "full_result.csv"

N_BOOT = 1000   # bootstrap resamples (games drawn with replacement)
SEED = 2026
MIN_SHARE_FOR_ALPHA = 0.05   # per-code alpha reported only for codes in at least 5% of the codings

print("Project root :", PROJECT_ROOT.name)
for path in [FORM_PATH, ASSIGNMENT_PATH, RESULT_PATH]:
    print(f"{str(path.relative_to(PROJECT_ROOT)):45s} exists: {path.exists()}")

## 2. Inputs

The coding is checked against the form and the assignment: every row comes from one of the two
experts assigned to its game, every coded feedback has exactly two codings, and every answer is an
option of the form. Games not yet coded by both experts are reported and left out of the analysis.

In [ ]:
form = json.load(open(FORM_PATH, encoding="utf-8"))
QUESTIONS = [q["id"] for q in form["questions"]]
OPTIONS = {q["id"]: [o["value"] for o in q["widget"]["options"]] for q in form["questions"]}
MULTI = [q["id"] for q in form["questions"] if q["widget"]["type"] == "CHECKBOX"]
EXCLUSIVE = {q["id"]: next((o["value"] for o in q["widget"]["options"] if o.get("exclusive")), None)
             for q in form["questions"]}

assignment = pd.read_csv(ASSIGNMENT_PATH, dtype=str, keep_default_na=False)
result = pd.read_csv(RESULT_PATH, dtype=str, keep_default_na=False)
experts_of = {g.game_id: (g.expert_1, g.expert_2) for g in assignment.itertuples()}


def codes(value: str) -> frozenset:
    return frozenset(v.strip() for v in value.split(",") if v.strip())


assert result.apply(lambda r: r.expert_id in experts_of[r.game_id], axis=1).all(), "coding by a non-assigned expert"
assert (result.groupby(["game_id", "trace_id"]).expert_id.nunique() == 2).all(), "feedback without two codings"
for col in QUESTIONS:
    given = set().union(*result[col].map(codes)) if col in MULTI else set(result[col])
    assert given <= set(OPTIONS[col]), (col, given - set(OPTIONS[col]))
    if EXCLUSIVE[col]:
        assert not result[col].map(lambda v: EXCLUSIVE[col] in codes(v) and len(codes(v)) > 1).any(), col

expected = assignment.set_index("game_id").n_received.astype(int)
coded = result.groupby("game_id").trace_id.nunique()
missing = expected[~expected.index.isin(coded.index)]
print(f"Coded games: {len(coded)} of {len(expected)} | feedback: {coded.sum()} of {expected.sum()}")
if len(missing):
    print("Not coded by both experts yet:", ", ".join(f"{g} ({n} feedback)" for g, n in missing.items()))
assert (coded == expected[coded.index]).all(), "partially coded game"

codings = result.expert_id.value_counts().rename("codings").to_frame()
codings["games"] = result.groupby("expert_id").game_id.nunique()
codings

## 3. Agreement measures

The two codings of each feedback are put side by side (expert A = `expert_1` of the assignment).
Measures used in the item sections:

- **Krippendorff's alpha** with three distances: nominal (two different answers are fully apart),
  ordinal (the distance between two stages grows with the number of codings lying between them),
  and MASI (Passonneau, 2006) for a whole multiple-choice answer, which gives partial credit when
  two sets of codes overlap. 1 = perfect agreement, 0 = agreement expected by chance.
- **Specific agreement** (Cicchetti and Feinstein, 1990) for one category: positive agreement is
  the share of the codings in this category matched by the other expert (2 x both / (2 x both +
  one only)); it stays readable for rare categories, where alpha is unstable.
- 95% confidence intervals by **cluster bootstrap over games** (percentile method): the games are
  drawn with replacement and every feedback of a drawn game is kept, so the dependence between the
  feedback of one game (same student, same expert pair) is preserved.

Each alpha is described with the Landis and Koch (1977) scale, defined for kappa and widely used
to describe agreement in words:

| Alpha | Agreement |
|---|---|
| 0.81 to 1.00 | almost perfect |
| 0.61 to 0.80 | substantial |
| 0.41 to 0.60 | moderate |
| 0.21 to 0.40 | fair |
| 0.00 to 0.20 | slight |
| below 0 | poor |

In [ ]:
def alpha(a, b, metric: str = "nominal", order: list | None = None) -> float:
    """Krippendorff's alpha for two codings per unit, no missing value."""
    values = list(a) + list(b)
    n = len(values)
    cats = list(dict.fromkeys(values)) if order is None else [c for c in order if c in set(values)]
    index = {c: i for i, c in enumerate(cats)}
    counts = np.bincount([index[v] for v in values], minlength=len(cats)).astype(float)
    if metric == "nominal":
        dist = 1.0 - np.eye(len(cats))
    elif metric == "ordinal":
        cum = np.concatenate([[0.0], np.cumsum(counts)])
        dist = np.array([[(cum[max(i, j) + 1] - cum[min(i, j)] - (counts[i] + counts[j]) / 2) ** 2
                          for j in range(len(cats))] for i in range(len(cats))])
    elif metric == "masi":
        dist = np.array([[masi_distance(x, y) for y in cats] for x in cats])
    else:
        raise ValueError(metric)
    observed = np.mean([dist[index[x], index[y]] for x, y in zip(a, b)])
    expected = counts @ dist @ counts / (n * (n - 1))
    return 1 - observed / expected if expected > 0 else np.nan


def masi_distance(x: frozenset, y: frozenset) -> float:
    if x == y:
        return 0.0
    monotonicity = 0.67 if x <= y or y <= x else (0.33 if x & y else 0.0)
    return 1 - len(x & y) / len(x | y) * monotonicity


def specific_agreement(x, y) -> tuple[float, float]:
    """Positive and negative specific agreement for one binary category."""
    x, y = np.asarray(x, bool), np.asarray(y, bool)
    both, none, one = (x & y).sum(), (~x & ~y).sum(), (x ^ y).sum()
    positive = 2 * both / (2 * both + one) if both + one else np.nan
    negative = 2 * none / (2 * none + one) if none + one else np.nan
    return positive, negative


rng = np.random.default_rng(SEED)
pairs = []
for (game_id, trace_id), group in result.groupby(["game_id", "trace_id"]):
    a, b = (group[group.expert_id == e].iloc[0] for e in experts_of[game_id])
    row = {"game_id": game_id, "trace_id": trace_id, "expert_a": a.expert_id, "expert_b": b.expert_id}
    for col in QUESTIONS:
        row[f"{col}_a"] = codes(a[col]) if col in MULTI else a[col]
        row[f"{col}_b"] = codes(b[col]) if col in MULTI else b[col]
    pairs.append(row)
pairs = pd.DataFrame(pairs)
pairs["group_id"] = pairs.game_id.map(dict(zip(assignment.game_id, assignment.group_id)))
games = pairs.game_id.unique()
rows_of_game = {g: np.flatnonzero(pairs.game_id.values == g) for g in games}
BOOT_ROWS = [np.concatenate([rows_of_game[g] for g in rng.choice(games, len(games))]) for _ in range(N_BOOT)]


def bootstrap_ci(stat, *columns) -> tuple[float, float]:
    """95% interval of stat(*columns) over resamples of whole games."""
    arrays = [np.asarray(c, dtype=object) for c in columns]
    values = [stat(*(arr[rows] for arr in arrays)) for rows in BOOT_ROWS]
    return tuple(np.nanpercentile(np.array(values, dtype=float), [2.5, 97.5]))


def with_ci(value: float, ci: tuple[float, float]) -> str:
    return f"{value:.3f} [{ci[0]:.3f}; {ci[1]:.3f}]"


def landis_koch_level(value: float) -> str:
    bands = [(0.80, "almost perfect (0.81 to 1.00)"), (0.60, "substantial (0.61 to 0.80)"),
             (0.40, "moderate (0.41 to 0.60)"), (0.20, "fair (0.21 to 0.40)"), (0.0, "slight (0.00 to 0.20)")]
    return next((label for bound, label in bands if value > bound), "slight (0.00 to 0.20)" if value == 0 else "poor (< 0)")


print(f"Feedback coded by two experts: {len(pairs)} in {len(games)} games "
      f"(B={int((pairs.group_id == 'B').sum())}, C={int((pairs.group_id == 'C').sum())}), "
      f"{pairs[['expert_a', 'expert_b']].apply(frozenset, axis=1).nunique()} expert pairs")

## 4. Resolution stage

Single-choice question on an ordered scale (S1-NONE to S6-COMP): the main measure is the **ordinal
alpha**, where a disagreement between neighbouring stages weighs less than one between distant
stages. The exact agreement, the share of feedback coded at most one stage apart and the nominal alpha
(every disagreement weighs the same) are given for comparison; the positive specific agreement per
stage and the confusion matrix show where the experts diverge.

In [ ]:
STAGE = "resolution_stage"
STAGES = OPTIONS[STAGE]
stage_a, stage_b = pairs[f"{STAGE}_a"].values, pairs[f"{STAGE}_b"].values
rank = {s: i for i, s in enumerate(STAGES)}
gap = np.abs(pd.Series(stage_a).map(rank).values - pd.Series(stage_b).map(rank).values)

ordinal = lambda a, b: alpha(a, b, "ordinal", STAGES)
nominal = lambda a, b: alpha(a, b, "nominal")
stage_alpha = ordinal(stage_a, stage_b)
stage_ci = bootstrap_ci(ordinal, stage_a, stage_b)
SUMMARY = [{"item": "Resolution stage", "type": "single choice, ordinal", "exact agreement": (gap == 0).mean(),
            "alpha": stage_alpha, "ci_low": stage_ci[0], "ci_high": stage_ci[1], "distance": "ordinal"}]

stage_nominal = nominal(stage_a, stage_b)
pd.DataFrame([
    {"measure": "Krippendorff's alpha, ordinal (main)", "value": with_ci(stage_alpha, stage_ci),
     "Landis and Koch": landis_koch_level(stage_alpha)},
    {"measure": "Krippendorff's alpha, nominal", "value": with_ci(stage_nominal, bootstrap_ci(nominal, stage_a, stage_b)),
     "Landis and Koch": landis_koch_level(stage_nominal)},
    {"measure": "Exact agreement", "value": f"{(gap == 0).mean():.0%}"},
    {"measure": "At most one stage apart", "value": f"{(gap <= 1).mean():.0%}"},
]).set_index("measure").fillna("")

In [ ]:
per_stage = []
for stage in STAGES:
    in_a, in_b = stage_a == stage, stage_b == stage
    positive, _ = specific_agreement(in_a, in_b)
    per_stage.append({"stage": stage, "codings": int(in_a.sum() + in_b.sum()),
                      "share of codings": f"{(in_a.sum() + in_b.sum()) / (2 * len(pairs)):.1%}", "both experts": int((in_a & in_b).sum()),
                      "one expert only": int((in_a ^ in_b).sum()), "positive agreement": round(positive, 2)})
display(pd.DataFrame(per_stage).set_index("stage"))

# Confusion matrix, symmetrized: the order of the two experts carries no meaning
confusion = pd.crosstab(pd.Categorical(stage_a, STAGES), pd.Categorical(stage_b, STAGES), dropna=False)
confusion = confusion + confusion.T - np.diag(np.diag(confusion))
confusion.index.name, confusion.columns.name = "stage (one expert)", "stage (other expert)"
confusion

### Interpretation

- The ordinal alpha (0.76, substantial) makes the resolution stage the most reliable item.
- The gap with the nominal alpha (0.48) shows that the experts share the model of progression: 94% of
  the feedback are coded at most one stage apart, and almost all disagreements are between neighbouring
  stages.
- The disagreements concentrate on three boundaries: S2/S3 (53 feedback), S4/S5 (33) and S3/S4 (30).
  S3-BLOC has the lowest positive agreement (0.52): deciding whether the naive approach has already
  been defeated by the limiting constraint is the hardest call of the scale.
- The extreme stages (S1-NONE, S6-COMP) are rare, since students seldom ask for help with an empty
  editor or a solved level.

## 5. Invalidity

Multiple-choice question where `R0-NONE` (valid content) excludes the error codes. The main measure
reduces the answer to **valid or invalid** (`R0-NONE` or at least one error code), the substantive
question of the item, with its nominal alpha. Invalid feedback is rare, which lowers alpha even when
raw agreement is high: the specific agreement on each answer and the number of feedback flagged by
one or both experts show where the disagreement lies. The alpha on the whole answer (MASI distance)
is given for comparison with the other multiple-choice items. The error codes are too rare for a
reliable alpha: only their counts and positive specific agreement are reported.

In [ ]:
INVALIDITY = "feedback_invalidity"
VALID = EXCLUSIVE[INVALIDITY]
answer_a, answer_b = pairs[f"{INVALIDITY}_a"].values, pairs[f"{INVALIDITY}_b"].values
invalid_a = np.array([VALID not in s for s in answer_a])
invalid_b = np.array([VALID not in s for s in answer_b])

masi = lambda a, b: alpha(a, b, "masi")
validity_alpha = nominal(invalid_a, invalid_b)
validity_ci = bootstrap_ci(nominal, invalid_a, invalid_b)
answer_alpha = masi(answer_a, answer_b)
answer_ci = bootstrap_ci(masi, answer_a, answer_b)
positive, negative = specific_agreement(invalid_a, invalid_b)
flagged_one, flagged_both = int((invalid_a | invalid_b).sum()), int((invalid_a & invalid_b).sum())
SUMMARY.append({"item": "Invalidity", "type": "valid / invalid, binary", "exact agreement": (invalid_a == invalid_b).mean(),
                "alpha": validity_alpha, "ci_low": validity_ci[0], "ci_high": validity_ci[1], "distance": "nominal"})

pd.DataFrame([
    {"measure": "Krippendorff's alpha, valid / invalid (main)", "value": with_ci(validity_alpha, validity_ci),
     "Landis and Koch": landis_koch_level(validity_alpha)},
    {"measure": "Krippendorff's alpha, whole answer (MASI)", "value": with_ci(answer_alpha, answer_ci),
     "Landis and Koch": landis_koch_level(answer_alpha)},
    {"measure": "Exact agreement, valid / invalid", "value": f"{(invalid_a == invalid_b).mean():.0%}"},
    {"measure": "Exact agreement, whole answer", "value": f"{(answer_a == answer_b).mean():.0%}"},
    {"measure": "Codings 'invalid' (share of codings)", "value": f"{invalid_a.sum() + invalid_b.sum()} ({(invalid_a.sum() + invalid_b.sum()) / (2 * len(pairs)):.1%})"},
    {"measure": "Specific agreement, invalid / valid", "value": f"{positive:.2f} / {negative:.2f}"},
    {"measure": "Feedback invalid for at least one expert", "value": f"{flagged_one} ({flagged_one / len(pairs):.0%})"},
    {"measure": "Feedback invalid for both experts", "value": f"{flagged_both} ({flagged_both / len(pairs):.0%})"},
]).set_index("measure").fillna("")

In [ ]:
per_code = []
for option in OPTIONS[INVALIDITY]:
    if option == VALID:
        continue
    in_a = np.array([option in s for s in answer_a])
    in_b = np.array([option in s for s in answer_b])
    positive_code, _ = specific_agreement(in_a, in_b)
    per_code.append({"code": option, "codings": int(in_a.sum() + in_b.sum()),
                     "share of codings": f"{(in_a.sum() + in_b.sum()) / (2 * len(pairs)):.1%}", "both experts": int((in_a & in_b).sum()),
                     "one expert only": int((in_a ^ in_b).sum()), "positive agreement": round(positive_code, 2)})
pd.DataFrame(per_code).set_index("code")

### Interpretation

- Raw agreement is high (81%) but alpha is only fair (0.28): invalid feedback is rare (15% of the
  codings), so agreement is mostly agreement on valid feedback.
- The specific agreements locate the disagreement: the experts agree when a feedback is valid
  (0.89), much less on which feedback are invalid (0.39). Of the 88 feedback flagged by at least one
  expert, only 21 are flagged by both, so the share of invalid feedback ranges from 6% to 25%
  depending on whether one or both experts are required.
- Judging validity requires checking the feedback against the student's code and actions: each
  expert detects only part of the errors, which makes the adjudication essential for this item.
- R4-ACTI (misreading of the student's activity) is the most frequent error code (60 codings, about
  half of the error codings), with a low positive agreement (0.33); the other error codes are too
  rare to assess their reliability.

## 6. Content

Multiple-choice question without exclusive option. The whole answer is measured with the **MASI
alpha**, the standard distance for sets of labels, which is the measure of the summary; it is severe
when the experts differ by a single code, so the nature of the disagreements is described (one
answer contained in the other, number of differing codes). Each code is then analysed as a binary
decision (ticked or not): counts, positive specific agreement and, for codes found in at least 5%
of the codings (`MIN_SHARE_FOR_ALPHA`), the nominal alpha.

In [ ]:
def per_code_report(col: str, skip: str | None = None) -> pd.DataFrame:
    """Binary agreement for each code of a multiple-choice question."""
    sets_a, sets_b = pairs[f"{col}_a"].values, pairs[f"{col}_b"].values
    rows = []
    for option in OPTIONS[col]:
        if option == skip:
            continue
        in_a = np.array([option in s for s in sets_a])
        in_b = np.array([option in s for s in sets_b])
        positive_code, _ = specific_agreement(in_a, in_b)
        share = (in_a.sum() + in_b.sum()) / (2 * len(pairs))
        row = {"code": option, "codings": int(in_a.sum() + in_b.sum()), "share of codings": f"{share:.1%}", "both experts": int((in_a & in_b).sum()),
               "one expert only": int((in_a ^ in_b).sum()), "positive agreement": round(positive_code, 2),
               "alpha [95% CI]": "", "Landis and Koch": ""}
        if share >= MIN_SHARE_FOR_ALPHA:
            value = nominal(in_a, in_b)
            row.update({"alpha [95% CI]": with_ci(value, bootstrap_ci(nominal, in_a, in_b)),
                        "Landis and Koch": landis_koch_level(value)})
        rows.append(row)
    return pd.DataFrame(rows).set_index("code")


def answer_report(col: str) -> tuple[pd.DataFrame, float, tuple[float, float]]:
    """Whole-answer agreement of a multiple-choice question and nature of its disagreements."""
    sets_a, sets_b = pairs[f"{col}_a"].values, pairs[f"{col}_b"].values
    value, ci = masi(sets_a, sets_b), bootstrap_ci(masi, sets_a, sets_b)
    differ = [(x, y) for x, y in zip(sets_a, sets_b) if x != y]
    nested = sum(x < y or y < x for x, y in differ)
    single = sum(len(x ^ y) == 1 for x, y in differ)
    n_codes = np.mean([len(s) for s in list(sets_a) + list(sets_b)])
    table = pd.DataFrame([
        {"measure": "Krippendorff's alpha, whole answer (MASI, main)", "value": with_ci(value, ci),
         "Landis and Koch": landis_koch_level(value)},
        {"measure": "Exact agreement", "value": f"{1 - len(differ) / len(pairs):.0%}"},
        {"measure": "Disagreements", "value": f"{len(differ)}"},
        {"measure": "  one answer contained in the other", "value": f"{nested} ({nested / len(differ):.0%})"},
        {"measure": "  a single code differs", "value": f"{single} ({single / len(differ):.0%})"},
        {"measure": "Codes per answer (mean)", "value": f"{n_codes:.1f}"},
    ]).set_index("measure").fillna("")
    return table, value, ci


CONTENT = "feedback_content"
content_table, content_alpha, content_ci = answer_report(CONTENT)
SUMMARY.append({"item": "Content", "type": "multiple choice", "exact agreement":
                (pairs[f"{CONTENT}_a"] == pairs[f"{CONTENT}_b"]).mean(),
                "alpha": content_alpha, "ci_low": content_ci[0], "ci_high": content_ci[1], "distance": "MASI"})
content_table

In [ ]:
per_code_report(CONTENT)

### Interpretation

- The whole-answer agreement is fair (MASI alpha 0.33, exact agreement 22%), but the disagreements
  are mostly about completeness rather than contradiction: in 65% of them one expert's answer
  contains the other's, and in 45% a single code differs.
- Per code, agreement is moderate (substantial for C3-CTAR). It is highest for the codes that rely on visible elements of the
  feedback: target and secondary concepts (C3-CTAR 0.60, C4-CSEC 0.58) and solution code
  (C6-SOLU 0.60). It is lower for the codes that require interpreting the feedback: task
  explanation versus narration of the game (C1-TASK 0.44), diagnosis (C5-DIAG 0.43) and limiting
  constraint (C2-CONS 0.42).
- Illustrating examples (C7-EXAM) and other content (C8-OTHE) are too rare to assess.

## 7. Instructions

Same analysis as the content: whole answer with the MASI alpha (measure of the summary), nature of
the disagreements, then each code as a binary decision. `I0-NONE` (no instruction) is the exclusive
option of the question; it is kept among the codes, since it is rare and does not carry a
substantive question of its own as `R0-NONE` does for the invalidity.

In [ ]:
INSTRUCTION = "feedback_instruction"
instruction_table, instruction_alpha, instruction_ci = answer_report(INSTRUCTION)
SUMMARY.append({"item": "Instructions", "type": "multiple choice", "exact agreement":
                (pairs[f"{INSTRUCTION}_a"] == pairs[f"{INSTRUCTION}_b"]).mean(),
                "alpha": instruction_alpha, "ci_low": instruction_ci[0], "ci_high": instruction_ci[1], "distance": "MASI"})
instruction_table

In [ ]:
per_code_report(INSTRUCTION)

### Interpretation

- The whole-answer agreement is moderate, at the upper limit of the range (MASI alpha 0.60, exact
  agreement 67%); as for the content, most disagreements bear on a single code (68%).
- This value averages two very different codes: the referral to the memo is almost perfectly
  identified (I2-MEMO 0.97), since it is explicit in the feedback, while the instruction to modify
  the code is only moderate (I1-CODE 0.49), since it is often implicit ("you need to store this
  value in a variable").
- The other instructions are rare; the referral to the level description keeps a good positive
  agreement (I3-DESC 0.70), unlike running the code (I4-EXEC 0.30) and other instructions
  (I5-OTHE 0.23).

## 8. Summary

In [ ]:
summary = pd.DataFrame(SUMMARY)
summary["alpha [95% CI]"] = [with_ci(r.alpha, (r.ci_low, r.ci_high)) for r in summary.itertuples()]
summary["exact agreement"] = summary["exact agreement"].map("{:.0%}".format)
summary["Landis and Koch"] = summary.alpha.map(landis_koch_level)
summary[["item", "type", "distance", "exact agreement", "alpha [95% CI]", "Landis and Koch"]].set_index("item")

### Interpretation

- The reliability of the grid follows the kind of judgement each item requires. The resolution stage,
  which places the student on a shared model of progression, is substantial (0.76). The instructions
  are moderate (0.60), the content and the invalidity fair (0.33 and 0.28).
- Within the items, agreement is high when a code relies on an element that is explicit in the
  feedback text (memo referral, solution code, named concept), moderate when it requires
  interpreting the intent of the feedback (task explanation, diagnosis, instruction to modify the
  code), and low when it requires checking the feedback against the student's activity (invalidity).
- On the multiple-choice items, the experts rarely contradict each other: most disagreements are a
  code ticked by one expert and not by the other.
- Codes found in fewer than 5% of the codings (all error codes except R4-ACTI, C7-EXAM, C8-OTHE,
  I0-NONE, I3-DESC, I4-EXEC, I5-OTHE) cannot be assessed on this corpus.

#### Consequences for the adjudication and the LLM evaluation

- **Adjudication.** 336 of the 359 feedback (94%) have at least one question in disagreement, for 615
  decisions in all. Both arbiters adjudicate every disagreement, keeping each time the answer of one
  of the two experts.
- **Gold standard.** It is built question by question: when the two arbiters disagree on a question,
  only this question is left without label.
- **LLM evaluation against the experts.** Each model is treated as one more annotator: its agreement
  with the experts is compared with the agreement between experts measured here. A model is at
  expert level on a measure when it does not fall more than 0.10 below the experts.
- **LLM evaluation against the gold standard.** Each model is scored with the macro-F1 of each item
  (mean F1 of its categories).
- Rare categories are left out of the conclusions.